# Crop Productivity Analysis 

## Data

The following datasets are utilized in this analysis for calculating and mapping crop productivity over the past years:

1. **Dynamic World Dataset**:
   - **Source**: [Dynamic World - Google and the World Resources Institute (WRI)](https://dynamicworld.app/) 
   - **Description**:The Dynamic World dataset provides a near real-time, high-resolution (10-meter) global land cover classification. It is derived from Sentinel-2 imagery and utilizes machine learning models to classify land cover into nine distinct classes, including water, trees, grass, crops, built areas, bare ground, shrubs, flooded vegetation, and snow/ice. The dataset offers data with minimal latency, enabling near-immediate analysis and decision-making.
   - **Spatial Resolution**: 10 meters.
   - **Temporal Coverage**: Data is available since mid-2015, updated continuously as Sentinel-2 imagery becomes available capturing near Real-time.

2. **MODIS Dataset**:
   - **Source:** NASA's Moderate Resolution Imaging Spectroradiometer [MODIS](https://modis.gsfc.nasa.gov/data/) on Terra and Aqua satellites.
   - **Description:** The MODIS dataset provides a wide range of data products, including land surface temperature, vegetation indices, and land cover classifications. It is widely used for monitoring and modeling land surface processes.
   - **Spatial Resolution:** 250 meters.
   - **Temporal Coverage:** Data is available from 2000 to the present, with daily to 16-day composite products.

3. **Administrative Boundaries (HDX)**:
   - **Source**: Humanitarian Data Exchange [HDX](https://data.humdata.org/).
   - **Description**: Geographic boundaries used for spatial aggregation and administrative analysis, such as calculating productivity metrics by region (e.g., governorate or district).
   - **Use Case**: The administrative boundaries are used to aggregate EVI statistics by region and facilitate reporting at various administrative levels.


In [1]:
from pathlib import Path
import ast


import ee
import geemap
from gee_zonal import ZonalStats
import geopandas as gpd
import pandas as pd
import plotnine as p9
import altair as alt
import numpy as np

PROJECT_ROOT = Path().cwd().parent.parent
DATA_PATH = PROJECT_ROOT / "data"
EVI_PATH = DATA_PATH / "EVI and Crop Land" / "EVI 2010-2025"
CROPLAND_PATH = DATA_PATH / "EVI and Crop Land" / "Crop Land"
BOUNDARIES_PATH = DATA_PATH / "boundaries"

ee.Initialize()
alt.data_transformers.disable_max_rows()

DataTransformerRegistry.enable('default')

In [2]:
def bitwiseExtract(value: ee.Image, fromBit: int, toBit: int | None = None) -> ee.Image:
    """Extract bits from a binary image."""
    toBit = fromBit if toBit is None else toBit
    maskSize = ee.Number(1).add(toBit).subtract(fromBit)
    mask = ee.Number(1).leftShift(maskSize).subtract(1)
    return value.rightShift(fromBit).bitwiseAnd(mask)


def apply_modisQA_mask(image: ee.Image):
    sqa = image.select("SummaryQA")
    dqa = image.select("DetailedQA")
    viQualityFlagsS = bitwiseExtract(sqa, 0, 1)
    viQualityFlagsD = bitwiseExtract(dqa, 0, 1)
    viSnowIceFlagsD = bitwiseExtract(dqa, 14)
    # Good data, use with confidence
    mask = (
        viQualityFlagsS.eq(0)
        .And(viQualityFlagsD.eq(0))
        .And(viQualityFlagsS.eq(1))
        .And(viQualityFlagsD.eq(1))
        .And(viSnowIceFlagsD)
        .eq(0)
    )
    return image.updateMask(mask)


def apply_scale_factor(image, scale_factor: float = 0.0001) -> ee.Image:
    """Apply scale factor to an image."""
    scaled_evi = image.multiply(scale_factor).copyProperties(
        image, ["system:time_start"]
    )
    return scaled_evi


def load_evi_data(
    start_date: str | ee.Date,
    end_date: str | ee.Date,
) -> ee.ImageCollection:
    """
    Load EVI data from MODIS.

    Args:
        start_date: Start date.
        end_date: End date.
        apply_crop_mask: Whether to apply crop mask.

    Returns:
        EVI image collection.
    """

    terra = (
        ee.ImageCollection("MODIS/061/MOD13Q1")
        .select(["EVI", "SummaryQA", "DetailedQA"])
        .filterDate(start_date, end_date)
    )

    aqua = (
        ee.ImageCollection("MODIS/061/MYD13Q1")
        .select(["EVI", "SummaryQA", "DetailedQA"])
        .filterDate(start_date, end_date)
    )

    mod13q1_QC = terra.map(apply_modisQA_mask)
    myd13q1_QC = aqua.map(apply_modisQA_mask)
    mxd13q1_cleaned = (
        mod13q1_QC.select("EVI").merge(myd13q1_QC.select("EVI")).map(apply_scale_factor)
    )
    mxd13q1 = mxd13q1_cleaned.sort("system:time_start")
    return mxd13q1


def filter_growing_season(
    df: pd.DataFrame, start_month: int = 7, end_month: int = 2
) -> pd.DataFrame:
    # Check if season spans two calendar years
    if end_month < start_month:
        months = list(range(start_month, 13)) + list(range(1, end_month + 1))
        season_data = df[df["date"].dt.month.isin(months)].copy()

        # Assign next year months to the previous year
        season_data["year"] = pd.to_datetime(
            season_data["date"].apply(
                lambda x: f"{x.year - 1}-01-01"
                if x.month <= end_month
                else f"{x.year}-01-01"
            )
        )
    else:
        months = list(range(start_month, end_month + 1))
        season_data = df[df["date"].dt.month.isin(months)].copy()

        season_data["year"] = pd.to_datetime(
            season_data["date"].dt.year.astype(str) + "-01-01"
        )

    return season_data

In [3]:
adm0_shp = gpd.read_file(BOUNDARIES_PATH / "eth_admbnda_adm0_csa_bofedb_itos_2021.shp")
adm1_shp = gpd.read_file(BOUNDARIES_PATH / "eth_admbnda_adm1_csa_bofedb_2021.shp")
adm2_shp = gpd.read_file(BOUNDARIES_PATH / "eth_admbnda_adm2_csa_bofedb_2021.shp")
adm3_shp = gpd.read_file(BOUNDARIES_PATH / "eth_admbnda_adm3_csa_bofedb_2021.shp")

adm0 = adm0_shp.pipe(geemap.geopandas_to_ee)
adm1 = adm1_shp.pipe(geemap.geopandas_to_ee)
adm2 = adm2_shp.pipe(geemap.geopandas_to_ee)
adm3 = adm3_shp.pipe(geemap.geopandas_to_ee)

start_date = ee.Date("2010-01-01")
end_date = ee.Date("2025-08-30")

evi_data = load_evi_data(start_date, end_date)

In [7]:
def create_cropland_mask_by_year(year: int):
    """
    Create cropland mask for a specific year using Dynamic World land cover.
    For years before 2015, uses 2015 mask as Dynamic World only starts in 2015.
    """
    cropland_value = 4

    # Dynamic World starts in 2015
    mask_year = max(year, 2015)

    yearly_start = ee.Date.fromYMD(mask_year, 1, 1)
    yearly_end = ee.Date.fromYMD(mask_year, 12, 31)

    yearly_lc = (
        ee.ImageCollection("GOOGLE/DYNAMICWORLD/V1")
        .filterDate(yearly_start, yearly_end)
        .filterBounds(adm0)
        .select("label")
        # Get the most frequent land cover class
        .reduce(ee.Reducer.mode())
    )

    crop_mask = yearly_lc.eq(cropland_value)
    return crop_mask


def create_evi_masked_by_year(year: int):
    """Create EVI collection masked to cropland for a specific year."""
    crop_mask = create_cropland_mask_by_year(year)

    evi_subset = evi_data.filterDate(
        ee.Date.fromYMD(year, 1, 1), ee.Date.fromYMD(year, 12, 31)
    )

    def apply_crop_mask(image):
        return image.updateMask(crop_mask).set("year", year)

    evi_masked = evi_subset.map(apply_crop_mask)
    return evi_masked

## Crop area statistics

In [ ]:
run = False
if run:
    # Split into 2 batches to avoid timeout
    total_features = len(adm1_shp)
    batch_size = int(np.ceil(total_features / 2))

    for year in range(2015, 2026):
        print(f"Processing year {year}...")

        for batch_num in range(2):
            start_idx = batch_num * batch_size
            end_idx = min((batch_num + 1) * batch_size, total_features)

            print(
                f"  Batch {batch_num + 1}/2 (regions {start_idx} to {end_idx - 1})..."
            )

            try:
                crop_year = create_cropland_mask_by_year(year)

                adm1_batch_gdf = adm1_shp.iloc[start_idx:end_idx]
                adm1_batch = adm1_batch_gdf.pipe(geemap.geopandas_to_ee)

                zs = ZonalStats(
                    ee_dataset=crop_year,
                    target_features=adm1_batch,
                    scale=10,
                    statistic_type="count",
                    output_dir="Ethiopia Crop Admin level 1",
                    output_name=f"ethiopia_adm1_cropland_stats_{year}_batch{batch_num + 1}",
                )

                zs.runZonalStats()
                print(f"  ✓ Completed batch {batch_num + 1}")

            except Exception as e:
                print(f"  ✗ Error processing batch {batch_num + 1}: {str(e)}")
                continue

        print(f"✓ Completed year {year}")

In [ ]:
run = False
if run:
    # Split into 3 batches to avoid timeout
    total_features = len(adm2_shp)
    batch_size = int(np.ceil(total_features / 3))

    for year in range(2015, 2026):
        print(f"Processing year {year}...")

        for batch_num in range(3):
            start_idx = batch_num * batch_size
            end_idx = min((batch_num + 1) * batch_size, total_features)

            print(
                f"  Batch {batch_num + 1}/3 (regions {start_idx} to {end_idx - 1})..."
            )
            try:
                crop_year = create_cropland_mask_by_year(year)
                adm2_batch_gdf = adm2_shp.iloc[start_idx:end_idx]
                adm2_batch = adm2_batch_gdf.pipe(geemap.geopandas_to_ee)

                zs = ZonalStats(
                    ee_dataset=crop_year,
                    target_features=adm2_batch,
                    scale=10,
                    statistic_type="count",
                    output_dir="Ethiopia Crop Admin level 2",
                    output_name=f"ethiopia_adm2_cropland_stats_{year}_batch{batch_num + 1}",
                )

                zs.runZonalStats()
                print(f"  ✓ Completed batch {batch_num + 1}")

            except Exception as e:
                print(f"✗ Error processing year {year}: {str(e)}")
                continue

        print(f"✓ Completed year {year}")

In [ ]:
run = False
if run:
    # Split into 8 batches to avoid timeout
    total_features = len(adm3_shp)
    batch_size = int(np.ceil(total_features / 8))

    for year in range(2015, 2026):
        print(f"Processing year {year}...")

        for batch_num in range(8):
            start_idx = batch_num * batch_size
            end_idx = min((batch_num + 1) * batch_size, total_features)

            print(
                f"  Batch {batch_num + 1}/8 (regions {start_idx} to {end_idx - 1})..."
            )
            try:
                crop_year = create_cropland_mask_by_year(year)
                adm3_batch_gdf = adm3_shp.iloc[start_idx:end_idx]
                adm3_batch = adm3_batch_gdf.pipe(geemap.geopandas_to_ee)

                zs = ZonalStats(
                    ee_dataset=crop_year,
                    target_features=adm3_batch,
                    scale=10,
                    statistic_type="count",
                    output_dir="Ethiopia Crop Admin level 3",
                    output_name=f"ethiopia_adm3_cropland_stats_{year}_batch{batch_num + 1}",
                )

                zs.runZonalStats()
                print(f"  ✓ Completed batch {batch_num + 1}")

            except Exception as e:
                print(f"✗ Error processing year {year}: {str(e)}")
                continue

        print(f"✓ Completed year {year}")

Processing year 2015...
  Batch 1/8 (regions 0 to 135)...
  ✓ Completed batch 1
  Batch 2/8 (regions 136 to 271)...
  ✓ Completed batch 2
  Batch 3/8 (regions 272 to 407)...
  ✓ Completed batch 3
  Batch 4/8 (regions 408 to 543)...
  ✓ Completed batch 4
  Batch 5/8 (regions 544 to 679)...
  ✓ Completed batch 5
  Batch 6/8 (regions 680 to 815)...
  ✓ Completed batch 6
  Batch 7/8 (regions 816 to 951)...
  ✓ Completed batch 7
  Batch 8/8 (regions 952 to 1081)...
  ✓ Completed batch 8
✓ Completed year 2015
Processing year 2016...
  Batch 1/8 (regions 0 to 135)...
  ✓ Completed batch 1
  Batch 2/8 (regions 136 to 271)...
  ✓ Completed batch 2
  Batch 3/8 (regions 272 to 407)...
  ✓ Completed batch 3
  Batch 4/8 (regions 408 to 543)...
  ✓ Completed batch 4
  Batch 5/8 (regions 544 to 679)...
  ✓ Completed batch 5
  Batch 6/8 (regions 680 to 815)...
  ✓ Completed batch 6
  Batch 7/8 (regions 816 to 951)...
  ✓ Completed batch 7
  Batch 8/8 (regions 952 to 1081)...
  ✓ Completed batch 8
✓ Co

In [ ]:
def load_dict(input_str):
    input_str = input_str.replace("null", "'null'")
    input_str = input_str.replace("=", ":")
    result_dict = ast.literal_eval(input_str)
    return result_dict


crop_count_files = CROPLAND_PATH / "Admin level 1" / "MIMU"
dfs = []
for file in crop_count_files.glob("*.csv"):
    df = pd.read_csv(file).assign(
        year=file.stem.split("_")[-2],
        histogram=lambda df: df["histogram"].apply(load_dict),
    )
    dfs.append(df)

dfs = pd.concat(dfs, ignore_index=True)
dfs_area = (
    dfs.join(pd.json_normalize(dfs["histogram"]))
    .rename(columns={"null": "Other", 1: "crop_area", 0: "non_crop"})
    .fillna(0)
    .drop(columns=["histogram"])
    .loc[
        :,
        [
            "year",
            "ST",
            "ST_PCODE",
            "crop_area",
        ],
    ]
    .sort_values(["crop_area"], ascending=False)
)

df_area = (
    dfs_area.assign(
        year=lambda df: df["year"].astype(str),
        # Each pixel is 10m x 10m = 100 m²; convert to hectares (1 ha = 10,000 m²)
        crop_area=lambda df: (df["crop_area"] * 100 / 10000),
    )
    .pivot(
        index=["ST", "ST_PCODE"],
        columns="year",
        values="crop_area",
    )
    .reset_index()
    .assign(
        pct_change_2015_2025=lambda df: (
            (df["2025"] - df["2015"]) / df["2015"] * 100
        ).round(2),
        abs_pct_change_2015_2025=lambda df: abs(df["pct_change_2015_2025"]),
    )
    .replace([float("inf"), np.nan], 0)
)

df_area

In [ ]:
(
    df_area.filter(
        [
            "ST",
            "ST_PCODE",
            "2015",
            "2025",
            "pct_change_2015_2025",
            "abs_pct_change_2015_2025",
        ]
    )
    .sort_values("2025", ascending=False)
    .rename(
        columns={
            "ST": "Name",
            "ST_PCODE": "PCODE",
            "2015": "Crop Area (ha) in 2015",
            "2025": "Crop Area (ha) in 2025",
            "pct_change_2015_2025": "% Change in Crop Area (2015-2025)",
            "abs_pct_change_2015_2025": "Absolute % Change in Crop Area (2015-2025)",
        }
    )
    .reset_index(drop=True)
)

In [ ]:
(
    alt.Chart(dfs_area, title="Crop Area in Myanmar by Admin Level 1 (2015-2025)")
    .mark_line(point=True)
    .encode(
        x="year:T",
        y="crop_area:Q",
        tooltip=["year:T", "crop_area:Q"],
        facet=alt.Facet("ST:N", columns=4, title=None),
    )
    .properties(width=160, height=100)
    .resolve_scale(y="independent")
)

## Crop Seasonality

Using this time series dataset of EVI images, we apply several pre-processing steps to extract critical phenological parameters: start of season (SOS), middle of season (MOS), end of season (EOS), length of season (LOS), etc. This workflow is heavily inspired by the [TIMESAT](https://web.nateko.lu.se/timesat/timesat.asp) software.

**Pre-processing steps**  
1. Remove outliers from dataset on per-pixel basis using median method: outlier if median from a moving window < or > standard deviation of time-series times 2.
2. Interpolate missing values linearly
3. Smooth data on per-pixel basis (using Savitsky Golay filter, window length of 3, and polyorder of 1)  

**Phenology Process**  
We then extract crop seasonality metrics using the seasonal amplitude method from the phenolopy package. 

The chart below shows the result of this process for a single crop pixel. The blue dots represent the raw EVI values, the black line represents the processed EVI values, and the dotted lines represent season parameters extracted for that pixel: start of season, peak of season, and end of season.

In [ ]:
run = False
if run:
    for year in range(2010, 2026):
        print(f"Processing year {year}...")

        try:
            evi_year = create_evi_masked_by_year(year)

            zs = ZonalStats(
                ee_dataset=evi_year,
                target_features=adm0,
                scale=250,
                statistic_type="median",
                output_dir="Admin level 0",
                output_name=f"ethiopia_adm0_evi_stats_{year}",
            )

            zs.runZonalStats()

            print(f"✓ Completed year {year}")

        except Exception as e:
            print(f"✗ Error processing year {year}: {str(e)}")
            continue

In [ ]:
def preprocess_evi(evi_file: str | Path) -> pd.DataFrame:
    """Preprocess EVI CSV file."""
    evi_df = pd.read_csv(evi_file)

    metadata_cols = [
        col
        for col in evi_df.columns
        for word in ["Name", "OBJECTID", "ST", "DT", "TS", "PCode", ".geo"]
        if word in col
    ]
    evi_df = (
        evi_df.rename(columns=lambda col: col[-14:] if col.endswith("_EVI") else col)
        .drop(columns=["system:index"])
        .melt(
            id_vars=metadata_cols,
            var_name="band_date",
            value_name="EVI",
        )
        .assign(
            date=lambda df: pd.to_datetime(
                df["band_date"].str.extract(r"(\d{4}_\d{2}_\d{2})")[0],
                format="%Y_%m_%d",
            ),
        )
    )
    return evi_df


evi_adm0_df = (
    pd.concat(
        [
            preprocess_evi(
                DATA_PATH
                / "EVI and Crop Land"
                / "EVI 2010-2025"
                / "Admin level 0"
                / "MIMU"
                / f"myanmar_adm0_evi_stats_{year}.csv"
            )
            for year in range(2010, 2026)
        ],
    )
    .drop(columns=[".geo"])
    .merge(adm0_shp.filter(["OBJECTID", "geometry"]), on="OBJECTID", how="left")
    .sort_values(["date"])
    .reset_index(drop=True)
    .assign(
        month=lambda df: df["date"].dt.to_period("M").dt.to_timestamp(),
        year=lambda df: df["date"].dt.to_period("Y").dt.to_timestamp(),
    )
    .set_index("date")
)

evi_adm0_df.head()

In [ ]:
from scipy.signal import savgol_filter


def preprocess_series(series):
    """TIMESAT-style preprocessing."""
    series = series.interpolate(limit_direction="both")  # Step 2: Interpolate
    median = series.rolling(window=5, center=True).median()
    std_dev = series.std()
    series = series.mask(
        (series - median).abs() > 2 * std_dev
    )  # Step 1: Remove outliers
    series = series.interpolate(limit_direction="both")
    smoothed = savgol_filter(series, window_length=5, polyorder=2)  # Step 3: Smooth
    return smoothed


def extract_sos_mos_eos(smoothed, dates, threshold=0.2):
    max_val = smoothed.max()
    min_val = smoothed.min()
    amp = max_val - min_val
    sos = mos = eos = None
    for i in range(1, len(smoothed)):
        if sos is None and smoothed[i] > min_val + threshold * amp:
            sos = dates[i]
        if smoothed[i] == max_val:
            mos = dates[i]
        if sos is not None and smoothed[i] < min_val + threshold * amp:
            eos = dates[i]
            break
    return sos, mos, eos

In [ ]:
seasonality_data = (
    evi_adm0_df.assign(month=lambda df: df.index.month)
    .groupby("month")
    .agg(EVI=("EVI", "mean"))
    # add several next months in the next year
    .pipe(lambda df: pd.concat([df, df.iloc[:6]]))
    .assign(smoothed=lambda df: preprocess_series(df["EVI"]))
    .assign(
        sos=lambda df: extract_sos_mos_eos(df["smoothed"].values, df.index)[0],
        mos=lambda df: extract_sos_mos_eos(df["smoothed"].values, df.index)[1],
        eos=lambda df: extract_sos_mos_eos(df["smoothed"].values, df.index)[2],
    )
    .iloc[:12]
    .assign(
        eos=lambda df: df["eos"].apply(lambda x: x - 12 if x and x > 12 else x),
        smoothed=lambda df: df["smoothed"].round(3),
        EVI=lambda df: df["EVI"].round(3),
    )
    .reset_index()
)

base = alt.Chart(seasonality_data)

points = base.mark_circle(color="steelblue", size=60).encode(
    x=alt.X(
        "month:O",
        scale=alt.Scale(domain=list(range(1, 13))),
        axis=alt.Axis(
            labelExpr="['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'][datum.value-1]",
            title="",
        ),
    ),
    y=alt.Y("EVI:Q"),
    tooltip=["month:O", "EVI:Q"],
)

line = base.mark_line(color="black").encode(
    x="month:O", y=alt.Y("smoothed:Q"), tooltip=["month:O", "smoothed:Q"]
)

sos_line = (
    base.mark_rule(strokeDash=[5, 5], color="gray")
    .encode(x="sos:O")
    .transform_filter(alt.expr.isValid(alt.datum.sos))
)

sos_text = (
    base.mark_text(
        align="center",
        baseline="bottom",
        dy=-5,
        fontSize=12,
        fontWeight="bold",
        color="gray",
    )
    .encode(x="sos:O", y=alt.Y("max(smoothed):Q"), text=alt.value("SOS"))
    .transform_filter(alt.expr.isValid(alt.datum.sos))
)

mos_line = (
    base.mark_rule(strokeDash=[5, 5], color="gray")
    .encode(x="mos:O")
    .transform_filter(alt.expr.isValid(alt.datum.mos))
)

mos_text = (
    base.mark_text(
        align="center",
        baseline="bottom",
        dy=-10,
        fontSize=12,
        fontWeight="bold",
        color="gray",
    )
    .encode(x="mos:O", y=alt.Y("max(smoothed):Q"), text=alt.value("MOS"))
    .transform_filter(alt.expr.isValid(alt.datum.mos))
)

eos_line = (
    base.mark_rule(strokeDash=[5, 5], color="gray")
    .encode(x="eos:O", text=alt.value("EOS"))
    .transform_filter(alt.expr.isValid(alt.datum.eos))
)

eos_text = (
    base.mark_text(
        align="center",
        baseline="bottom",
        dy=-5,
        fontSize=12,
        fontWeight="bold",
        color="gray",
    )
    .encode(x="eos:O", y=alt.Y("max(smoothed):Q"), text=alt.value("EOS"))
    .transform_filter(alt.expr.isValid(alt.datum.eos))
)

chart = (
    (points + line + sos_line + sos_text + mos_line + mos_text + eos_line + eos_text)
    .properties(title="Crop Seasonality based on EVI", width=600, height=400)
    .interactive()
)

chart

Based on the phenology process, we identified the seasonality to start in July and end in February with the peak being in October. This can vary with geographic region and crop type as well, however, that has not been taken into consideration in this version.

In [ ]:
monthly_evi = (
    evi_adm0_df.groupby(pd.Grouper(freq="MS"))
    .agg(EVI=("EVI", "median"))
    .assign(month=lambda x: x.index.month, year=lambda x: x.index.year)
)

(
    p9.ggplot(monthly_evi)
    + p9.aes(x="month", y="EVI", group="year")
    + p9.geom_line(alpha=0.5)
    + p9.theme_minimal()
    + p9.scale_x_continuous(
        breaks=range(1, 13),
        labels=[
            "Jan",
            "Feb",
            "Mar",
            "Apr",
            "May",
            "Jun",
            "Jul",
            "Aug",
            "Sep",
            "Oct",
            "Nov",
            "Dec",
        ],
    )
    + p9.labs(title="Monthly Median EVI in Myanmar (2010-2025)", x="")
)

The following figure shows the median EVI in Myanmar from 2010 to 2025 during the crop growing season (February to June). The shaded area represents the interquartile range (IQR) of EVI values across all pixels, indicating the variability in vegetation health during this period.

In [ ]:
evi_adm0_median = (
    evi_adm0_df.reset_index()
    .pipe(filter_growing_season)
    .groupby("year", as_index=False)
    .agg(EVI=("EVI", "median"))
    .assign(EVI=lambda df: df["EVI"].round(3))
)

(
    alt.Chart(evi_adm0_median, title="Median EVI in Myanmar (2010-2025)")
    .mark_line(point=True)
    .encode(
        x=alt.X("year:T", title=""),
        y=alt.Y("EVI:Q"),
        tooltip=["year:T", "EVI:Q"],
    )
    .properties(title="Median EVI in Myanmar (2010-2025)", width=600, height=300)
    .interactive()
)

## Trends in EVI

### Admin level 1

In [ ]:
run = False
if run:
    # Split into 2 batches to avoid timeout
    total_features = len(adm1_shp)
    batch_size = int(np.ceil(total_features / 2))

    for year in range(2010, 2026):
        print(f"Processing year {year}...")

        for batch_num in range(2):
            start_idx = batch_num * batch_size
            end_idx = min((batch_num + 1) * batch_size, total_features)

            print(
                f"  Batch {batch_num + 1}/2 (regions {start_idx} to {end_idx - 1})..."
            )

            try:
                evi_year = create_evi_masked_by_year(year)

                adm1_batch_gdf = adm1_shp.iloc[start_idx:end_idx]
                adm1_batch = adm1_batch_gdf.pipe(geemap.geopandas_to_ee)

                zs = ZonalStats(
                    ee_dataset=evi_year,
                    target_features=adm1_batch,
                    scale=250,
                    statistic_type="median",
                    output_dir="Admin level 1",
                    output_name=f"ethiopia_adm1_evi_stats_{year}_batch{batch_num + 1}",
                )

                zs.runZonalStats()
                print(f"  ✓ Completed batch {batch_num + 1}")

            except Exception as e:
                print(f"  ✗ Error processing batch {batch_num + 1}: {str(e)}")
                continue

        print(f"✓ Completed year {year}")

In [ ]:
evi_adm1_df = (
    pd.concat(
        [
            preprocess_evi(
                DATA_PATH
                / "EVI and Crop Land"
                / "EVI 2010-2025"
                / "Admin level 1"
                / "MIMU"
                / f"myanmar_adm1_evi_stats_{year}_batch{batch_num}.csv"
            )
            for year in range(2010, 2026)
            for batch_num in range(1, 4)
        ],
    )
    .drop(columns=[".geo"])
    .merge(adm1_shp.filter(["ST_PCODE", "geometry"]), on="ST_PCODE", how="left")
    .sort_values(["date"])
    .reset_index(drop=True)
)
evi_adm1_df.head()

The figure below shows the trends of median EVI from 2010-2025 on admin level 1.

In [ ]:
evi_adm1_median = (
    evi_adm1_df.pipe(filter_growing_season)
    .groupby(["year", "ST_PCODE", "ST"], as_index=False)
    .agg(EVI=("EVI", "median"))
    .assign(EVI=lambda df: df["EVI"].round(3))
    .rename(columns={"ST": "Name"})
)


(
    alt.Chart(
        evi_adm1_median, title="Median EVI in Myanmar by Admin Level 1 (2010-2025)"
    )
    .mark_line(point=True)
    .encode(
        x="year:T",
        y="EVI:Q",
        tooltip=["year:T", "EVI:Q"],
        facet=alt.Facet("Name:N", columns=4, title=None),
    )
    .properties(width=160, height=100)
)

The figure below shows a choropleth maps of EVI for each admin level 1 from 2010-2025

In [ ]:
(
    alt.Chart(
        evi_adm1_median.assign(year=lambda df: df["year"].dt.year),
        title="Median EVI in Myanmar by Admin Level 1 (2010-2025)",
    )
    .mark_geoshape(stroke="white", strokeWidth=1.5)
    .encode(
        shape="geo:G",
        color="EVI:Q",
        tooltip=["Name:N", "EVI:Q"],
        facet=alt.Facet("year:N", columns=4, title=None),
    )
    .transform_lookup(
        lookup="ST_PCODE",
        from_=alt.LookupData(data=adm1_shp, key="ST_PCODE"),
        as_="geo",
    )
    .properties(width=200, height=200)
)

The figure below shows the percent change of EVI compared to previous year from 2010-2025 on admin level 1.

In [ ]:
evi_adm1_pct_change = evi_adm1_median.sort_values(["ST_PCODE", "year"]).assign(
    EVI_pct_change=lambda df: df.groupby(["ST_PCODE", "Name"])["EVI"].pct_change(),
)
(
    alt.Chart(
        evi_adm1_pct_change,
        title="EVI Percentage Change in Myanmar by Admin Level 1 (2010-2025)",
    )
    .mark_line(point=True)
    .encode(
        x="year:T",
        y=alt.Y(
            "EVI_pct_change:Q", axis=alt.Axis(format=".1%"), title="Percentage Change"
        ),
        tooltip=[
            "year:T",
            alt.Tooltip("EVI_pct_change:Q", format=".1%", title="EVI % Change"),
        ],
        facet=alt.Facet("Name:N", columns=4, title=None),
    )
    .properties(width=160, height=100)
)

### Admin level 2

In [ ]:
run = False
if run:
    # Split into 3 batches to avoid timeout
    total_features = len(adm2_shp)
    batch_size = int(np.ceil(total_features / 3))

    for year in range(2010, 2026):
        print(f"Processing year {year}...")

        for batch_num in range(3):
            start_idx = batch_num * batch_size
            end_idx = min((batch_num + 1) * batch_size, total_features)

            print(
                f"  Batch {batch_num + 1}/3 (regions {start_idx} to {end_idx - 1})..."
            )

            try:
                evi_year = create_evi_masked_by_year(year)

                adm2_batch_gdf = adm2_shp.iloc[start_idx:end_idx]
                adm2_batch = adm2_batch_gdf.pipe(geemap.geopandas_to_ee)

                zs = ZonalStats(
                    ee_dataset=evi_year,
                    target_features=adm2_batch,
                    scale=250,
                    statistic_type="median",
                    output_dir="Admin level 2",
                    output_name=f"ethiopia_adm2_evi_stats_{year}_batch{batch_num + 1}",
                )

                zs.runZonalStats()
                print(f"  ✓ Completed batch {batch_num + 1}")

            except Exception as e:
                print(f"  ✗ Error processing batch {batch_num + 1}: {str(e)}")
                continue

        print(f"✓ Completed year {year}")

In [ ]:
evi_adm2_df = (
    pd.concat(
        [
            preprocess_evi(
                DATA_PATH
                / "EVI and Crop Land"
                / "EVI 2010-2025"
                / "Admin level 2"
                / "MIMU"
                / f"myanmar_adm2_evi_stats_{year}_batch{batch_num}.csv"
            )
            for year in range(2010, 2026)
            for batch_num in range(1, 6)
        ],
    )
    .drop(columns=[".geo"])
    .merge(adm2_shp.filter(["DT_PCODE", "geometry"]), on="DT_PCODE", how="left")
    .sort_values(["date", "DT"])
)
evi_adm2_df.head()

In [ ]:
evi_adm2_median = (
    evi_adm2_df.pipe(filter_growing_season)
    .groupby(["year", "DT_PCODE", "DT"], as_index=False)
    .agg(EVI=("EVI", "median"))
    .assign(EVI=lambda df: df["EVI"].round(3))
    .rename(columns={"DT": "Name"})
)

options = sorted(evi_adm2_median["Name"].unique().tolist())
input_dropdown = alt.binding_select(options=options, name="Region ")
selection = alt.selection_point(fields=["Name"], value=options[0], bind=input_dropdown)

base = alt.Chart(
    evi_adm2_median, title="Median EVI in Myanmar by Admin Level 2 (2010-2025)"
).encode(x="year:T", y="EVI:Q", detail="Name:N")

background = base.mark_line(point=True).encode(
    color=alt.value("lightgrey"), opacity=alt.value(0.5)
)

highlight = (
    base.mark_line(point=True, strokeWidth=2)
    .encode(
        color=alt.value("steelblue"),
        opacity=alt.value(1),
        tooltip=["year:T", "Name:N", "EVI:Q"],
    )
    .transform_filter(selection)
)

((background + highlight).add_params(selection).properties(width=500, height=300))

The figure below shows a choropleth maps of EVI for each admin level 2 from 2010-2025

In [ ]:
(
    alt.Chart(
        evi_adm2_median.assign(year=lambda df: df["year"].dt.year).fillna({"EVI": 0}),
        title="Median EVI in Myanmar by Admin Level 2 (2010-2025)",
    )
    .mark_geoshape(stroke="white", strokeWidth=1)
    .encode(
        shape="geo:G",
        color=alt.condition("datum.EVI !== 0", "EVI:Q", alt.value("lightgray")),
        tooltip=["Name:N", "EVI:Q"],
        facet=alt.Facet("year:N", columns=4),
    )
    .transform_lookup(
        lookup="DT_PCODE",
        from_=alt.LookupData(data=adm2_shp, key="DT_PCODE"),
        as_="geo",
    )
    .properties(width=250, height=250)
    .interactive()
)

### Admin level 3

In [ ]:
run = False
if run:
    # Split into 8 batches to avoid timeout
    total_features = len(adm3_shp)
    batch_size = int(np.ceil(total_features / 8))

    for year in range(2010, 2026):
        print(f"Processing year {year}...")

        for batch_num in range(8):
            start_idx = batch_num * batch_size
            end_idx = min((batch_num + 1) * batch_size, total_features)

            print(
                f"  Batch {batch_num + 1}/8 (regions {start_idx} to {end_idx - 1})..."
            )

            try:
                evi_year = create_evi_masked_by_year(year)

                adm3_batch_gdf = adm3_shp.iloc[start_idx:end_idx]
                adm3_batch = adm3_batch_gdf.pipe(geemap.geopandas_to_ee)

                zs = ZonalStats(
                    ee_dataset=evi_year,
                    target_features=adm3_batch,
                    scale=250,
                    statistic_type="median",
                    output_dir="Admin level 3",
                    output_name=f"ethiopia_adm3_evi_stats_{year}_batch{batch_num + 1}",
                )

                zs.runZonalStats()
                print(f"  ✓ Completed batch {batch_num + 1}")

            except Exception as e:
                print(f"  ✗ Error processing batch {batch_num + 1}: {str(e)}")
                continue

        print(f"✓ Completed year {year}")

Processing year 2010...
  Batch 1/8 (regions 0 to 135)...
  ✓ Completed batch 1
  Batch 2/8 (regions 136 to 271)...
  ✓ Completed batch 2
  Batch 3/8 (regions 272 to 407)...
  ✓ Completed batch 3
  Batch 4/8 (regions 408 to 543)...
  ✓ Completed batch 4
  Batch 5/8 (regions 544 to 679)...
  ✓ Completed batch 5
  Batch 6/8 (regions 680 to 815)...
  ✓ Completed batch 6
  Batch 7/8 (regions 816 to 951)...
  ✓ Completed batch 7
  Batch 8/8 (regions 952 to 1081)...
  ✓ Completed batch 8
✓ Completed year 2010
Processing year 2011...
  Batch 1/8 (regions 0 to 135)...
  ✓ Completed batch 1
  Batch 2/8 (regions 136 to 271)...
  ✓ Completed batch 2
  Batch 3/8 (regions 272 to 407)...
  ✓ Completed batch 3
  Batch 4/8 (regions 408 to 543)...
  ✓ Completed batch 4
  Batch 5/8 (regions 544 to 679)...
  ✓ Completed batch 5
  Batch 6/8 (regions 680 to 815)...
  ✓ Completed batch 6
  Batch 7/8 (regions 816 to 951)...
  ✓ Completed batch 7
  Batch 8/8 (regions 952 to 1081)...
  ✓ Completed batch 8
✓ Co

In [ ]:
evi_adm3_df = (
    pd.concat(
        [
            preprocess_evi(
                DATA_PATH
                / "EVI and Crop Land"
                / "EVI 2010-2025"
                / "Admin level 3"
                / "MIMU"
                / f"myanmar_adm3_evi_stats_{year}_batch{batch_num}.csv"
            )
            for year in range(2010, 2026)
            for batch_num in range(1, 6)
        ],
    )
    .drop(columns=[".geo"])
    .merge(adm3_shp.filter(["TS_PCODE", "geometry"]), on="TS_PCODE", how="left")
    .sort_values(["date", "TS"])
)
evi_adm3_df

In [ ]:
evi_adm3_median = (
    evi_adm3_df.pipe(filter_growing_season)
    .groupby(["year", "TS_PCODE", "TS"], as_index=False)
    .agg(EVI=("EVI", "median"))
    .assign(EVI=lambda df: df["EVI"].round(3))
    .rename(columns={"TS": "Name"})
)

options = sorted(evi_adm3_median["Name"].unique().tolist())
input_dropdown = alt.binding_select(options=options, name="Region ")
selection = alt.selection_point(fields=["Name"], bind=input_dropdown, value=options[0])

base = alt.Chart(
    evi_adm3_median, title="Median EVI in Myanmar by Admin Level 3 (2010-2025)"
).encode(x="year:T", y="EVI:Q", detail="Name:N")

background = base.mark_line(point=True).encode(
    color=alt.value("lightgrey"), opacity=alt.value(0.3)
)

highlight = (
    base.mark_line(point=True, strokeWidth=2)
    .encode(
        color=alt.value("steelblue"),
        opacity=alt.value(1),
        tooltip=["year:T", "Name:N", "EVI:Q"],
    )
    .transform_filter(selection)
)

((background + highlight).add_params(selection).properties(width=500, height=300))

The figure below shows a choropleth maps of EVI for each admin level 3 from 2010-2025

In [ ]:
def complete_cases(
    df: pd.DataFrame, group_cols: list[str], value_col: str
) -> pd.DataFrame:
    """Make sure all combinations of group_cols are present, and fill missing values with 0."""
    complete_df = (
        pd.MultiIndex.from_product(
            [df[col].unique() for col in group_cols],
            names=group_cols,
        )
        .to_frame(index=False)
        .merge(df, on=group_cols, how="left")
        .fillna({value_col: 0})
    )
    return complete_df


evi_adm3_median = (
    evi_adm3_df.pipe(filter_growing_season)
    .assign(year=lambda df: df["year"].dt.year)
    .groupby(["TS", "TS_PCODE", "year"], as_index=False)
    .agg(EVI=("EVI", "median"))
    .assign(EVI=lambda df: df["EVI"].round(3))
    .rename(columns={"TS": "Name"})
    .pipe(complete_cases, group_cols=["TS_PCODE", "year"], value_col="EVI")
)

(
    alt.Chart(
        evi_adm3_median, title="Median EVI in Myanmar by Admin Level 3 (2010-2025)"
    )
    .mark_geoshape(stroke="white", strokeWidth=0.5)
    .encode(
        shape="geo:G",
        color=alt.condition("datum.EVI !== 0", "EVI:Q", alt.value("lightgray")),
        tooltip=["Name:N", "EVI:Q"],
        facet=alt.Facet("year:N", columns=4, title=None),
    )
    .transform_lookup(
        lookup="TS_PCODE",
        from_=alt.LookupData(data=adm3_shp, key="TS_PCODE"),
        as_="geo",
    )
    .properties(width=170, height=170)
    .interactive()
)